In [ ]:
from yeager_utils import *
import ssapy

In [ ]:
t0 = Time("2025-1-1")

In [ ]:
surface_rv(lat=0, lon=-90, t=t0), surface_rv_ssapy(lat=0, lon=-90, t=t0)

In [ ]:
orbit = Orbit.fromKeplerianElements(a=RGEO, e=.4, i=0, pa=0, raan=0, trueAnomaly=np.radians(0), t=t0)

rs, v = rv(orbit, np.arange(0, orbit.period, 600))
orbit_plot(rs, show=True)


print("Complete.")

In [ ]:
t0 = Time("2025-1-1T12:00:00.000", scale='utc')
times = get_times(duration=(12, 'hour'), freq=(1, 's'), t0=t0).gps

kwargs = dict(
    mass=250,  # [kg] --> was 1e4
    area=0.25,  # [m^2]
    CD=2.3,  # Drag coefficient
    CR=1.3,  # Radiation pressure coefficient
)

earth = ssapy.get_body("earth", model='egm2008')
moon = ssapy.get_body("moon")
sun = ssapy.get_body("sun")

# Accelerations - pass a body object or string of body name.
aEarth = ssapy.AccelKepler() + ssapy.AccelHarmonic(earth, 180, 180)
aMoon = ssapy.AccelThirdBody(moon) + ssapy.AccelHarmonic(moon)
aSun = ssapy.AccelThirdBody(sun)
aSolRad = ssapy.AccelSolRad()
aEarthRad = ssapy.AccelEarthRad()
aDrag = ssapy.AccelDrag()
accel = aEarth + aMoon + aSun + aSolRad + aEarthRad + aDrag

a = RGEO
e = 0
i = 0
pa = 0
raan = 0
trueAnomaly = 0

kElements = [a, e, i, pa, raan, trueAnomaly]

orbit = ssapy.Orbit.fromKeplerianElements(*kElements, t0, propkw=kwargs)
orbital_period = orbit.period / (60 * 60 * 24)  # in days

# Define the time indices for the maneuver in the times array
t1_index = 0  # Replace with the actual index for the start time
t2_index = 100  # Replace with the actual index for the end time
print(f"Maneuver window: {Time(times[t1_index], format='gps').ymdhms} to {Time(times[t2_index], format='gps').ymdhms}")

# Example acceleration in NTW coordinates for the maneuver
thrust = -1  # m/s
accel_T = accel + ssapy.AccelConstNTW(accelntw=[0.0, thrust, 0.0], time_breakpoints=[times[t1_index], times[t2_index]])
accel_N = accel + ssapy.AccelConstNTW(accelntw=[thrust, 0, 0.0], time_breakpoints=[times[t1_index], times[t2_index]])

rs = []
propagators = [accel, accel_N, accel_T]

try:
    for acc in propagators:
        r, _ = ssapy.rv(orbit, times, propagator=SciPyPropagator(acc))
        rs.append(r)

    r, _ = leapfrog(orbit.r, orbit.v, times, velocity=(t1_index, t2_index, thrust))
    rs.append(r)
except RuntimeError as err:
    print(err)



gamma, heading = calc_gamma_and_heading(r2, times)
orbit_plot(rs, times, show=True)